# 1.0 — Выбор источника данных (одно место)

**Это единственное место, где переключается датасет всего пайплайна.** Доступны два
взаимозаменяемых источника одного формата:

| Источник | Что это | Чем собирается |
|---|---|---|
| `synthetic` | синтетическая популяция, кривые PPR(N) гладкие по построению | генерируется здесь же |
| `real_objects` | реальные объекты: пиклы + ведомость, **гладкая линия PPR** по верхней огибающей | ноутбук `1_1_3` |

Выбранный источник копируется в **канонический каталог `data/demo_run`**, который читают все
последующие ноутбуки (анализ `1_2`–`1_4`, обучение `2_*`, оценка `3_*`). Менять остальные
ноутбуки не нужно — поменяли `SOURCE` здесь, выполнили ноутбук, и весь пайплайн работает на
выбранных данных.

Гладкая линия PPR (верхняя огибающая квазисинусоиды + монотонное сглаживание) реализована в
`liquefaction_ai.data.smooth_ppr_trajectory` и применяется загрузчиком реальных объектов.

## Окружение

In [2]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

import os
from liquefaction_ai import get_default_config, set_global_seed
from liquefaction_ai.data import (available_sources, materialize_dataset,
                                  load_active_population, prepare_benchmark_dataset)

config = get_default_config()
set_global_seed(config.seed)

# Быстрый режим для проверки (LIQ_QUICK=1): уменьшает синтетическую популяцию
if os.environ.get("LIQ_QUICK"):
    from dataclasses import replace
    config = replace(config, n_scenarios=1200, benchmark_subset=800)
print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /sessions/determined-cool-fermat/mnt/liquefaction-ai


## Шаг 1. Какие источники уже собраны

`synthetic` собирается на месте при выборе. Реальные источники нужно предварительно собрать
ноутбуками `1_1_3` / `1_1_4` (тогда напротив них будет «готов = True»).

In [3]:
rows = []
for name, ready, path in available_sources(REPO_ROOT):
    rows.append({"источник": name, "готов": ready, "каталог": str(path.relative_to(REPO_ROOT))})
display(pd.DataFrame(rows))

,источник,готов,каталог
0,synthetic,True,data/demo_source_synthetic
1,real_objects,True,data/real_objects


## Шаг 2. ВЫБОР ИСТОЧНИКА

Поставьте нужное значение `SOURCE` и выполните ноутбук. Допустимо:
`"synthetic"`, `"real_objects"`.

In [4]:
# >>> ЕДИНСТВЕННАЯ НАСТРОЙКА <<<
SOURCE = os.environ.get("LIQ_DATASET", "real_objects")   # synthetic | real_objects

canonical = materialize_dataset(SOURCE, REPO_ROOT, config)
print(f"Активный источник: {SOURCE}")
print(f"Скопировано в каноническ. каталог: {canonical}")
print("Все последующие ноутбуки (1_2–1_4, 2_*, 3_*) теперь читают эти данные.")

Активный источник: real_objects
Скопировано в каноническ. каталог: /sessions/determined-cool-fermat/mnt/liquefaction-ai/data/demo_run
Все последующие ноутбуки (1_2–1_4, 2_*, 3_*) теперь читают эти данные.


## Шаг 3. Проверка активного датасета

In [5]:
import torch
population, loaded_config = load_active_population(REPO_ROOT)
benchmark = prepare_benchmark_dataset(population, loaded_config, torch.device("cpu"))
meta = population["meta"]
for split_name in ["train", "val", "test"]:
    s = benchmark[split_name]
    print(f"{split_name:6s}: static={tuple(s['static'].shape)}, seq={tuple(s['seq_in'].shape)}, "
          f"liq_rate={float(s['label'].float().mean()):.3f}")
display(pd.DataFrame({
    "Показатель": ["источник", "образцов", "доля разжижения", "средний N_liq",
                   "медианный пик PPR", "длина seq", "статич. признаков"],
    "Значение": [SOURCE, len(meta), round(float(meta["liq_label"].mean()), 4),
                 round(float(meta["N_liq_true"].mean()), 1),
                 round(float(meta["PPR_max_true"].median()), 4),
                 population["r_obs"].shape[1], population["static_features"].shape[1]],
}))

train : static=(466, 34), seq=(466, 72, 5), liq_rate=0.607
val   : static=(99, 34), seq=(99, 72, 5), liq_rate=0.606
test  : static=(101, 34), seq=(101, 72, 5), liq_rate=0.614


,Показатель,Значение
0,источник,real_objects
1,образцов,666
2,доля разжижения,0.6081
3,средний N_liq,2429.1
4,медианный пик PPR,1.003
5,длина seq,72
6,статич. признаков,34


## Шаг 4. Кривые PPR(N) активного датасета

In [6]:
import plotly.graph_objects as go
from liquefaction_ai.viz import save_figure

r_obs = population["r_obs"]; cyc = population["cycles"]; vmask = population["valid_mask"]
rng = np.random.default_rng(loaded_config.seed)
idx = rng.choice(len(meta), size=min(12, len(meta)), replace=False)
fig = go.Figure()
for i in idx:
    m = vmask[i] > 0
    fig.add_trace(go.Scatter(x=cyc[i][m], y=r_obs[i][m], mode="lines", showlegend=False,
                             hovertemplate="N=%{x:.0f}, PPR=%{y:.2f}<extra></extra>"))
fig.add_hline(y=1.0, line_dash="dash", line_color="red", annotation_text="PPR=1")
fig.update_layout(title=f"PPR(N) — активный источник: {SOURCE}",
                  xaxis_title="Число циклов N", yaxis_title="PPR, д.е.", height=440)
if SAVE_FIGS:
    save_figure(fig, f"1_0_active_dataset_{SOURCE}")
fig.show()

## Итог

Активный датасет лежит в `data/demo_run`. Чтобы переключиться — поменяйте `SOURCE` в шаге 2 и
выполните ноутбук заново. Источник `real_objects` предварительно соберите ноутбуком
`1_1_3_real_objects_loader`.

Дальше идите по обычному пути: `1_2` (анализ) → `1_4` (сплиты) → `2_*` (обучение) →
`3_*` (оценка). Все они работают поверх `data/demo_run` без изменений.